# IPL Match Winner Prediction
Predicting the winner of an IPL match based on pre-match features using Logistic Regression.

In [231]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

## 1. Load the Cleaned Data

In [232]:
# Load the interim cleaned matches data
matches = pd.read_csv('data/interim/matches_clean.csv')
print(f"Total matches: {len(matches)}")
matches.head(3)

Total matches: 636


,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2
0,1,2017,Hyderabad,2017-04-05,SRH,RCB,RCB,field,normal,0,SRH,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong
1,2,2017,Pune,2017-04-06,MI,RPS,RPS,field,normal,0,RPS,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi
2,3,2017,Rajkot,2017-04-07,GL,KKR,KKR,field,normal,0,KKR,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan


## 2. Preprocessing

In [233]:
# Filter out matches with 'No Result'
data = matches[matches['winner'] != 'No Result'].copy()

# Select features available before the match
features = ['team1', 'team2', 'toss_winner', 'toss_decision', 'venue']
target = 'winner'

X = data[features]
y = data[target]

# Convert categorical variables into numerical dummy variables (One-Hot Encoding)
X_encoded = pd.get_dummies(X, columns=features)
print(f"Features after encoding: {X_encoded.shape[1]}")

Features after encoding: 76


## 3. Train/Test Split

In [234]:
# Split 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

Training set size: 506
Testing set size: 127


## 4. Train the Model

In [235]:
# Initialize and train the Logistic Regression model
# model = LogisticRegression(max_iter=1000)  # Old Model
model = RandomForestClassifier(n_estimators=180,max_depth=20 ,random_state=42)  # New Model
model.fit(X_train, y_train)
print("Model training complete!")

Model training complete!


## 5. Evaluation

In [236]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

# Print detailed classification report for top teams
print("Classification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 58.27%

Classification Report:
              precision    recall  f1-score   support

         CSK       0.57      0.55      0.56        22
         DCG       0.25      0.33      0.29         6
          DD       0.58      0.78      0.67         9
          GL       1.00      0.33      0.50         3
         KKR       0.75      0.79      0.77        19
         KTK       0.00      0.00      0.00         1
        KXIP       0.67      0.50      0.57        16
          MI       0.56      0.56      0.56        16
         PWI       1.00      0.50      0.67         2
         RCB       0.60      0.56      0.58        16
         RPS       1.00      0.67      0.80         3
          RR       0.30      0.38      0.33         8
         SRH       0.62      0.83      0.71         6

    accuracy                           0.58       127
   macro avg       0.61      0.52      0.54       127
weighted avg       0.61      0.58      0.58       127



## 6. Pre-Match Predictor Function

In [237]:
def predict_prematch_winner(team1, team2, toss_winner, toss_decision, venue):
    # Create input dataframe
    input_data = pd.DataFrame({
        'team1': [team1],
        'team2': [team2],
        'toss_winner': [toss_winner],
        'toss_decision': [toss_decision],
        'venue': [venue]
    })
    
    # One-hot encode using the exact same columns as the training data
    input_encoded = pd.get_dummies(input_data, columns=['team1', 'team2', 'toss_winner', 'toss_decision', 'venue'])
    
    # Add missing dummy columns filled with 0s to match training data shape
    for col in X_encoded.columns:
        if col not in input_encoded.columns:
            input_encoded[col] = 0
            
    # Ensure column order matches exactly
    input_encoded = input_encoded[X_encoded.columns]
    
    # Predict
    prediction = model.predict(input_encoded)[0]
    prob = model.predict_proba(input_encoded)[0]
    
    print(f"--- PRE-MATCH SCENARIO ---")
    print(f"{team1} vs {team2} at {venue}")
    print(f"Toss: {toss_winner} elected to {toss_decision}")
    print(f"--------------------------")
    print(f"Predicted Winner: {prediction}")
    
    # Find indices for the two playing teams to print their specific probabilities
    classes = list(model.classes_)
    if team1 in classes and team2 in classes:
        t1_idx = classes.index(team1)
        t2_idx = classes.index(team2)
        print(f"Probability ({team1}): {prob[t1_idx]*100:.1f}%")
        print(f"Probability ({team2}): {prob[t2_idx]*100:.1f}%")

# Let's test a hypothetical scenario:
# Mumbai Indians vs Chennai Super Kings at Wankhede Stadium. CSK wins toss and elects to field.
predict_prematch_winner(
    team1='MI', 
    team2='CSK', 
    toss_winner='CSK', 
    toss_decision='field', 
    venue='Wankhede Stadium'
)

--- PRE-MATCH SCENARIO ---
MI vs CSK at Wankhede Stadium
Toss: CSK elected to field
--------------------------
Predicted Winner: CSK
Probability (MI): 24.0%
Probability (CSK): 66.5%
